In [1]:
import openslide
from PIL import Image
import os
import json
import numpy as np
import cv2
import math
import matplotlib.pyplot as plt

In [2]:
wsi_path = '/mnt/sda/datasets/TCGA_raw/TCGA/WSI/BRCA_svs'

In [3]:
with open('/home/liesgame/project/RL/SlideReasoner/datasets/summary/BRCA_report_v2.json', 'r', encoding='utf-8') as f:
    BRCA_report_v2 = json.load(f)
with open('/home/liesgame/project/RL/SlideReasoner/datasets/summary/BRCA_without_report_case_ids.json', 'r', encoding='utf-8') as f:
   BRCA_without_report_case_ids = json.load(f)

In [ ]:
import math
IMAGE_FACTOR = 16 * 2# 宽和高最后得是28的倍数
MIN_PIXELS = 4 * IMAGE_FACTOR * IMAGE_FACTOR# 最小像素数，这是宽*高，等宽高的情况下，宽和高都是2*28
MAX_PIXELS = 16384 * IMAGE_FACTOR * IMAGE_FACTOR# 最大像素数，这是宽*高，等宽高的情况下，宽和高都是128*28
MAX_RATIO = 200# 最大宽和高的比例，比如宽=1，高是200

VIDEO_MIN_PIXELS = 128 * IMAGE_FACTOR * IMAGE_FACTOR# 视频的最小像素数，这是宽*高
VIDEO_MAX_PIXELS = 768 * IMAGE_FACTOR * IMAGE_FACTOR# 视频的最大像素数，这是宽*高
# VIDEO_TOTAL_PIXELS = 24576 * 28 * 28
VIDEO_TOTAL_PIXELS = 128000 * IMAGE_FACTOR * IMAGE_FACTOR * 0.9# 视频的总像素数，28*28作为一个token，即可以有128000*0.9=115200个token
"""
这里看起来115200个token数好多啊，简单算一下
如果video的单张图片对应的像素数到了最大，也就是768 * 28 * 28
那单张图片占了768个token
所以最多可以从视频里拿出115200/768=150张图片
代码里的实现逻辑是先找到取多少帧，然后再算每张图片有多少像素点
"""
FRAME_FACTOR = 2# 每秒视频里可以取到几张图片
FPS = 2.0
FPS_MIN_FRAMES = 4# 最小取4帧
FPS_MAX_FRAMES = 768# 最大取768帧
def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    # 返回离number最近的可整除factor的整数
    # 四舍五入
    # round_by_factor(3000, 28)=2996
    # round_by_factor(6, 28)=0
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    # 返回大于等于number且能被factor整除的最小值
    # 向上取
    # ceil_by_factor(5,28)=28
    # ceil_by_factor(29,28)=56
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    # 返回小于等于number且能被factor整除的最大值
    # 向下取
    # floor_by_factor(5,28)=0
    # floor_by_factor(29,28)=28
    return math.floor(number / factor) * factor
def smart_resize(
    height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS
) -> tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:
    1. Both dimensions (height and width) are divisible by 'factor'.
    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].
    3. The aspect ratio of the image is maintained as closely as possible.
    """
    # smart_resize(6,6) = (56, 56) 56*56==4*28*28
    # smart_resize(30006,30006) = (3584, 3584) 最大尺寸是16384 * 28 * 28
    # smart_resize(30006,300006) = (1120, 11312) 保持长宽比的情况下达到最大尺寸
    # smart_resize(3000, 2000) = (2996, 1988) 正常尺寸，能够整除28
    # 长宽比要小于MAX_RATIO，即200
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))# 最小为factor
    w_bar = max(factor, round_by_factor(width, factor))# 最小为factor

    reszied_beta = 1
    if h_bar * w_bar > max_pixels:# 16384 * 28 * 28
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
        reszied_beta = beta
    elif h_bar * w_bar < min_pixels:# 
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
        reszied_beta = 1 / beta
    return h_bar, w_bar, reszied_beta

def convert_to_qwen25vl_format(bbox, orig_height, orig_width, factor=IMAGE_FACTOR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS):
    new_height, new_width, reszied_beta = smart_resize(orig_height, orig_width, factor, min_pixels, max_pixels)
    scale_w = new_width / orig_width
    scale_h = new_height / orig_height
    
    x1, y1, x2, y2 = bbox
    x1_new = round(x1 * scale_w)
    y1_new = round(y1 * scale_h)
    x2_new = round(x2 * scale_w)
    y2_new = round(y2 * scale_h)
    
    # 这个位置可以设置error    
    x1_new = max(0, min(x1_new, new_width - 1))
    y1_new = max(0, min(y1_new, new_height - 1))
    x2_new = max(0, min(x2_new, new_width - 1))
    y2_new = max(0, min(y2_new, new_height - 1))

    return [x1_new, y1_new, x2_new, y2_new]

def revert_from_qwen25vl_format(scaled_bbox, orig_height, orig_width, factor=IMAGE_FACTOR, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS):
    """
    将经过缩放的 bounding box 还原到原始图像坐标系。

    Args:
        scaled_bbox: 缩放后的 bbox，格式为 [x1, y1, x2, y2]。
        orig_height: 原始图像的高度。
        orig_width: 原始图像的宽度。
        factor, min_pixels, max_pixels: 与 `smart_resize` 中使用的参数必须完全相同。

    Returns:
        还原后的 bbox，格式为 [x1, y1, x2, y2]。
    """
    # 1. 使用原始尺寸重新计算出缩放后的尺寸，以确保得到相同的缩放比例
    new_height, new_width, reszied_beta = smart_resize(orig_height, orig_width, factor, min_pixels, max_pixels)
    
    # 2. 计算宽和高的缩放比例
    #    为避免除以零的错误，进行安全检查
    if orig_width == 0 or orig_height == 0:
        return [0, 0, 0, 0]
        
    scale_w = new_width / orig_width
    scale_h = new_height / orig_height

    if scale_w == 0 or scale_h == 0:
        return [0, 0, 0, 0]

    # 3. 将缩放后的坐标除以缩放比例，进行逆向操作
    x1_new, y1_new, x2_new, y2_new = scaled_bbox
    
    x1_orig = round(x1_new / scale_w)
    y1_orig = round(y1_new / scale_h)
    x2_orig = round(x2_new / scale_w)
    y2_orig = round(y2_new / scale_h)
    
    # 4. (可选但推荐) 将还原后的坐标限制在原始图像边界内
    # 这个位置可以设置error
    x1_orig = max(0, min(x1_orig, orig_width - 1))
    y1_orig = max(0, min(y1_orig, orig_height - 1))
    x2_orig = max(0, min(x2_orig, orig_width - 1))
    y2_orig = max(0, min(y2_orig, orig_height - 1))
    
    return [x1_orig, y1_orig, x2_orig, y2_orig]

In [5]:
def get_roi_at_mpp_optimized(
    slide: openslide.OpenSlide,
    source_roi: tuple[int, int, int, int],
    source_mpp: float,
    source_level0_x: int,
    source_level0_y: int,
    target_mpp: float,
    level0_mpp: float,
    min_pixels: int = 32
) -> tuple[Image.Image, tuple[int, int, int, int]]:
    """
    严格禁止升采样。如果目标MPP高于原生分辨率,则输出原生最高分辨率的图像。
    返回图像、其在Level 0的ROI坐标,以及图像的最终有效MPP。
    """
    # 1. 获取基础MPP和所有层级的MPP
    native_mpp = level0_mpp
    width, height = slide.dimensions



    # 2. 将源ROI换算到Level 0坐标 (所有计算的基准)
    scale_factor_to_level0 = source_mpp / native_mpp
    src_x, src_y, src_w, src_h = source_roi
    src_x, src_y, src_w, src_h = source_roi
    if src_x < 0 or src_y < 0 or src_w <= 0 or src_h <= 0 or source_level0_x <0 or source_level0_y < 0 :
        raise ValueError(
            f"source_level0_x: {source_level0_x}, source_level0_y: {source_level0_y}, src_x: {src_x}, src_y: {src_y}, src_w: {src_w} and src_h: {src_h} must be larger than 0"
        )
    level0_x = math.floor(src_x * scale_factor_to_level0) + source_level0_x
    level0_y = math.floor(src_y * scale_factor_to_level0) + source_level0_y
    level0_w = math.floor(src_w * scale_factor_to_level0)
    level0_h = math.floor(src_h * scale_factor_to_level0)
    output_level0_roi = (level0_x, level0_y, level0_w, level0_h)

    if level0_x + level0_w> width or level0_y + level0_h > height:
        raise ValueError(
            f"level0_x: {level0_x} + level0_w: {level0_w} = {level0_x + level0_w} be smaller than width: {width}, and level0_y: {level0_y} + level0_h: {level0_h} = {level0_y + level0_h} be smaller than height: {height}"
        )


    # 3. 【核心限制逻辑】根据目标MPP决定最终尺寸和读取层级
    if target_mpp < native_mpp:
        # --- 情况A: 请求的分辨率过高，执行限制 ---
        # logging.warning(f"目标MPP {target_mpp:.4f} 高于最大分辨率 {native_mpp:.4f}。将输出最大分辨率图像。")
        print(f"目标MPP {target_mpp:.4f} 高于最大分辨率 {native_mpp:.4f}。将输出最大分辨率图像 {native_mpp:.4f}。")
        
        # 目标尺寸被限制在Level 0的尺寸
        target_w = max(min_pixels, level0_w)
        target_h = max(min_pixels, level0_h)
        # 最终的有效MPP是原生MPP
        effective_mpp = native_mpp
        # 必须从Level 0读取
        optimal_level = 0
        
        # 确保尺寸至少为1
        optimal_level_w = max(min_pixels, level0_w)
        optimal_level_h = max(min_pixels, level0_h)

    else:
        # --- 情况B: 请求的分辨率有效，执行优化逻辑 ---
        effective_mpp = target_mpp
        level_mpps = [native_mpp * ds for ds in slide.level_downsamples]
        valid_levels = [i for i, mpp in enumerate(level_mpps) if mpp <= target_mpp]
        optimal_level = valid_levels[-1] # 选择最接近的层级
        
        # logging.info(f"目标MPP: {target_mpp:.2f} -> 智能选择从 Level {optimal_level} (MPP: {level_mpps[optimal_level]:.2f}) 读取数据。")
        print(f"目标MPP: {target_mpp:.2f} -> 智能选择从 Level {optimal_level} (MPP: {level_mpps[optimal_level]:.2f}) 读取数据。")

        optimal_level_downsample = slide.level_downsamples[optimal_level]
        optimal_level_w = max(min_pixels, math.floor(level0_w / optimal_level_downsample))
        optimal_level_h = max(min_pixels, math.floor(level0_h / optimal_level_downsample))
        
        scale_factor_to_target = native_mpp / target_mpp
        target_w = max(min_pixels, math.floor(level0_w * scale_factor_to_target))
        target_h = max(min_pixels, math.floor(level0_h * scale_factor_to_target))

    print(f"level0_w: {level0_w}, level0_h:{level0_h} -> optimal_level_w: {optimal_level_w}, optimal_level_h: {optimal_level_h} -> target_w: {target_w},  target_h: {target_h} 。")
    intermediate_image = slide.read_region(
        (level0_x, level0_y),
        optimal_level,
        (optimal_level_w, optimal_level_h)
    )
    # 4. 进行最后的微调缩放（如果需要）
    if intermediate_image.size != (target_w, target_h):
        target_image = intermediate_image.resize((target_w, target_h), Image.Resampling.LANCZOS)
    else:
        target_image = intermediate_image
    
    return target_image.convert("RGB"), output_level0_roi, effective_mpp, level0_w, level0_h, optimal_level_w, optimal_level_h, target_w, target_h

In [11]:
target_mpp = 10.0
data_path = '/home/liesgame/project/RL/SlideReasoner/datasets'
path_name = "resized_image_qwen3vl/mpp_10"
resize_path = os.path.join(data_path, path_name)
resize_image_path = os.path.join(resize_path, 'image')
resize_meta_path = os.path.join(resize_path, 'meta')
os.makedirs(resize_image_path, exist_ok=True)
os.makedirs(resize_meta_path, exist_ok=True)
BRCA_without_mpp = []
length_index = 0
BRCA_slide_info = {}
all_count = len([i for i in BRCA_report_v2.keys() if i not in BRCA_without_report_case_ids])
for i in BRCA_report_v2.keys():
    wsi_file_name = BRCA_report_v2[i]['wsi_file_name']
    wsi_file_path = os.path.join(wsi_path, wsi_file_name)
    slide = openslide.OpenSlide(wsi_file_path)
    width, height = slide.dimensions
    properties = slide.properties
    if properties.get('openslide.mpp-x') == None :
        print(f"wsi_file_name: {wsi_file_name} lost MPP")
        BRCA_without_mpp.append(wsi_file_name)
        continue
    BRCA_slide_info[i] = {}
    print(f"wsi_file_name: {wsi_file_name}")
    # 获取基本信息
    print("Levels (缩放层级数量):", slide.level_count)  # 金字塔层级数
    print("Dimensions (最大分辨率尺寸):", slide.dimensions)  # 最高层级的宽高 (width, height)
    print("Level Dimensions (各层级尺寸):", slide.level_dimensions)  # 所有层级的尺寸列表
    print("Level Downsamples (各层级下采样倍数):", slide.level_downsamples)  # 各层级相对于level 0的下采样比例


    BRCA_slide_info[i]['wsi_file_name'] = wsi_file_name
    BRCA_slide_info[i]['Levels'] = slide.level_count
    BRCA_slide_info[i]['Dimensions'] = [slide.dimensions]
    BRCA_slide_info[i]['Level_Dimensions'] = [list(i) for i in slide.level_dimensions]
    BRCA_slide_info[i]['Level_Downsamples'] = [ i for i in slide.level_downsamples]
    
    BRCA_slide_info[i]['level0_mpp'] = float(properties['openslide.mpp-x'])
    level0_mpp = float(properties['openslide.mpp-x'])
    print(level0_mpp)
    BRCA_slide_info[i]['Level_MPP'] = [i * level0_mpp for i in slide.level_downsamples]


    traget_image, output_level0_roi, output_mpp, level0_w, level0_h, optimal_level_w, optimal_level_h, target_w, target_h = get_roi_at_mpp_optimized(
        slide = slide,
        source_roi = (0, 0, width, height),
        source_mpp = level0_mpp,
        source_level0_x = 0,
        source_level0_y = 0,
        target_mpp = target_mpp,
        level0_mpp = level0_mpp
    )
    BRCA_slide_info[i]['target_mpp'] = target_mpp
    BRCA_slide_info[i]['output_mpp'] = output_mpp
    BRCA_slide_info[i]['level0_w'] = level0_w
    BRCA_slide_info[i]['level0_h'] = level0_h
    BRCA_slide_info[i]['optimal_level_w'] = optimal_level_w
    BRCA_slide_info[i]['optimal_level_h'] = optimal_level_h
    BRCA_slide_info[i]['target_w'] = target_w
    BRCA_slide_info[i]['target_h'] = target_h


    print(f'output_mpp: {output_mpp}')
    mpp_width, mpp_height = traget_image.size
    resized_height, resized_width, reszied_beta = smart_resize(
        height = mpp_height, 
        width = mpp_width
    )

    BRCA_slide_info[i]['mpp_width'] = mpp_width
    BRCA_slide_info[i]['mpp_height'] = mpp_height

    BRCA_slide_info[i]['mpp_image_token'] = math.ceil(mpp_width * mpp_height / IMAGE_FACTOR / IMAGE_FACTOR)
    BRCA_slide_info[i]['reszied_beta'] = reszied_beta
    BRCA_slide_info[i]['resized_width'] = resized_width
    BRCA_slide_info[i]['resized_height'] = resized_height
    BRCA_slide_info[i]['reszied_beta_times_output_mpp'] = reszied_beta * output_mpp
    print(f'resized_width: {resized_width}, resized_height: {resized_height}')
    print(f'reszied_beta * output_mpp: {reszied_beta * output_mpp}')
    length_index += 1
    print('current count {} / {}'.format(length_index, all_count))
    with open(os.path.join(resize_meta_path, i + '.json'), 'w', encoding='utf-8') as f:
        json.dump(BRCA_slide_info[i], f, ensure_ascii=False, indent=4)

    traget_image.resize((resized_width, resized_height), Image.Resampling.LANCZOS).save(os.path.join( resize_image_path , i + ".png"), compress_level=6, optimize=False)
with open(os.path.join(resize_path, 'meta_all.json'), 'w', encoding='utf-8') as f:
    json.dump(BRCA_slide_info, f, ensure_ascii=False, indent=4)

wsi_file_name: TCGA-BH-A0EB-01Z-00-DX1.70D7BBA0-214D-4D1A-933C-7CFCDA5416A3.svs
Levels (缩放层级数量): 4
Dimensions (最大分辨率尺寸): (136935, 85556)
Level Dimensions (各层级尺寸): ((136935, 85556), (34233, 21389), (8558, 5347), (2139, 1336))
Level Downsamples (各层级下采样倍数): (1.0, 4.000043817369205, 16.000783015577966, 64.02857748738148)
0.2485
目标MPP: 10.00 -> 智能选择从 Level 2 (MPP: 3.98) 读取数据。
level0_w: 136935, level0_h:85556 -> optimal_level_w: 8558, optimal_level_h: 5346 -> target_w: 3402,  target_h: 2126 。
output_mpp: 10.0
resized_width: 3392, resized_height: 2112
reszied_beta * output_mpp: 10.0
current count 1 / 933
wsi_file_name: TCGA-EW-A6SB-01Z-00-DX1.D56E1922-01A9-4AEE-AB95-D69447DD13EE.svs
Levels (缩放层级数量): 4
Dimensions (最大分辨率尺寸): (115536, 75222)
Level Dimensions (各层级尺寸): ((115536, 75222), (28884, 18805), (7221, 4701), (3610, 2350))
Level Downsamples (各层级下采样倍数): (1.0, 4.00005317734645, 16.00063816209317, 32.00689691754583)
0.2525
目标MPP: 10.00 -> 智能选择从 Level 3 (MPP: 8.08) 读取数据。
level0_w: 115536, level

In [ ]:
thumbnail_root = '/home/liesgame/project/RL/SlideReason/Resource/datasets/resized_image_mpp_5_meta/'

In [ ]:
roi1 = [11, 182, 24, 3089]
convert_roi = convert_to_qwen25vl_format(
    bbox = roi1,
    orig_height = mpp_height,
    orig_width = mpp_width
)
print(convert_roi)
revert_roi = revert_from_qwen25vl_format(
    scaled_bbox = convert_roi,
    orig_height = mpp_height,
    orig_width = mpp_width
)
print(revert_roi)

In [ ]:
traget_image.size

In [ ]:
level0_mpp

In [ ]:
traget_image.size

In [ ]:
output_level0_roi

In [ ]:
slide.properties

In [ ]:
wsi_file_name = BRCA_report_mapping['TCGA-B6-A0I1']['wsi_file_name']

In [ ]:
wsi_file_path = os.path.join(wsi_path, wsi_file_name)

In [ ]:
slide = openslide.OpenSlide(wsi_file_path)

In [ ]:
width, height = slide.dimensions
print(f"Level 0 dimensions: {width} x {height}")

In [ ]:
# 获取基本信息
print("Levels (缩放层级数量):", slide.level_count)  # 金字塔层级数
print("Dimensions (最大分辨率尺寸):", slide.dimensions)  # 最高层级的宽高 (width, height)
print("Level Dimensions (各层级尺寸):", slide.level_dimensions)  # 所有层级的尺寸列表
print("Level Downsamples (各层级下采样倍数):", slide.level_downsamples)  # 各层级相对于level 0的下采样比例


In [ ]:
properties = slide.properties
print(properties)
level0_mpp = float(properties['openslide.mpp-x'])
print(level0_mpp)

In [ ]:
level3_downsample = slide.level_downsamples[3]
print(level3_downsample)
source_mpp = level0_mpp * level3_downsample
print(source_mpp)

In [ ]:
slide.level_dimensions[3]

In [ ]:
slide.level_dimensions[3][0] * slide.level_dimensions[3][1] / 28 / 28 

In [ ]:
slide.read_region(
    (0, 0),
    3,
    slide.level_dimensions[3]
)

In [ ]:
slide.read_region(
    (56644, 43582),
    2,
    (1000, 1000)
)

In [ ]:
level0_mpp * slide.level_downsamples[3]

In [ ]:
traget_image, output_level0_roi, output_mpp = get_roi_at_mpp_optimized(
    slide = slide,
    source_roi = (0, 0, 1000, 1200),
    source_mpp = 8.0,
    source_level0_x = 56644,
    source_level0_y = 43582,
    target_mpp = 8.0,
    level0_mpp = level0_mpp
)

In [ ]:
output_level0_roi

In [ ]:
output_mpp

In [ ]:
traget_image

In [ ]:
traget_image.

In [ ]:
traget_image.mode

In [ ]:
traget_image.size

In [ ]:
smart_resize(1199, 999)

In [ ]:
traget_image.size

In [ ]:
traget_image, output_level0_roi, output_mpp = get_roi_at_mpp_optimized(
    slide = slide,
    source_roi = (0, 0, 500, 600),
    source_mpp = 8.0,
    source_level0_x = 56644,
    source_level0_y = 43582,
    target_mpp = 4.0,
    level0_mpp = level0_mpp
)

In [ ]:
slide.level_dimensions[3]

In [ ]:
traget_image

In [ ]:
traget_image.size

In [ ]:
origin_roi_x1_y1_x2_y2 = [150, 300, 650, 700]

In [ ]:
x1, y1, x2, y2 = origin_roi_x1_y1_x2_y2

In [ ]:
resized_height, resized_width = smart_resize(
    height = 1199, 
    width = 999
)
print(resized_height, resized_width)

In [ ]:
smart_resize(
    height = resized_height, 
    width = resized_width
)

In [ ]:
show_rectangle(
    image_narrary = np.array(traget_image),
    roi = origin_roi_x1_y1_x2_y2
)

In [ ]:
show_rectangle(
    image_narrary = np.array(traget_image.resize((resized_width, resized_height))),
    roi = convert_roi
)

In [ ]:
traget_image.resize((resized_width, resized_height))

In [ ]:
convert_roi = convert_to_qwen25vl_format(
    bbox = origin_roi_x1_y1_x2_y2,
    orig_height = traget_image.size[1],
    orig_width = traget_image.size[0]
)
convert_roi

In [ ]:
revert_roi = revert_from_qwen25vl_format(
    scaled_bbox = convert_roi,
    orig_height = traget_image.size[1],
    orig_width = traget_image.size[0]
)
revert_roi

In [ ]:
def show_rectangle(
    image_narrary: np.ndarray, 
    roi: tuple[int, int, int, int],
    color: tuple[int, int, int] = (0, 255, 0),
    line_pixel: int = 3,
    label: str = 'test',
    label_up: int = 10
):

    """
        (0, 255, 0)      # 纯绿色
        (0, 100, 0)      # 暗绿色
        (255, 0, 0)      # 纯蓝色
        (100, 0, 0)      # 暗蓝色
        (0, 255, 255)    # 纯黄色
        (0, 150, 150)    # 中等黄色
        (128, 128, 128)  # 中灰色
        (0, 0, 0)        # 黑色
        roi = x1, y1, x2, y2
    """
    x1, y1, x2, y2 = roi
    image = image_narrary.copy()
    cv2.rectangle(image, (x1, y1), (x2, y2), color, line_pixel)
    cv2.putText(image, label, (x1, y1 - label_up), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, line_pixel)
    image = cv2.cvtColor(traget_image_qwen_resize_narray, cv2.COLOR_BGR2RGB)
    plt.imshow(image)
    plt.axis('off')
    plt.show()